# Exercise 17.1: Simulating the FitzHugh-Nagumo model

We will now upgrade our bistable FDM solver to solve the full FHN model. We add the new parameters for the recovery variable: $\epsilon = 0.005$ and $\gamma = 2.0$.


## Exercise 17.1a: Adding the recovery variable

Extend your vectorized solver from the previous chapter to include the $w$ variable. You will need to calculate the reaction term for both $V$ and $w$ at every time step.

_(Note: $w$ does not diffuse physically through space, so there is no spatial second derivative term for $w$!)_


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

# Parameters
k = 2.0
A = 1.0
alpha = 0.1
L = 100
eps = 0.005
gamma = 2.0

dx = 1
dt = 0.1
N = int(L / dx)
n_steps = 1400
save_every = 10

# Initialize arrays
v = np.zeros(N + 1)
w = np.zeros(N + 1)

# Apply stimulus to the left edge
left = int(N / 10)
v[:left] = 0.3

# Slicing arrays for internal nodes
I = np.arange(1, N)
Ip = I + 1
Im = I - 1

snapshots_v = [v.copy()]
snapshots_w = [w.copy()]

for i in range(n_steps):
    # Safely copy previous states
    v_prev = np.copy(v)
    w_prev = np.copy(w)

    # 1. Calculate reaction terms
    I_ion_v = ...
    I_ion_w = ...

    # 2. Add diffusion to v
    v[I] = ...

    # 3. Update boundary nodes for v
    v[0] = ...
    v[N] = ...

    # 4. Add reaction terms to update v and w
    v = ...
    w = ...

    if (i + 1) % save_every == 0:
        snapshots_v.append(v.copy())
        snapshots_w.append(w.copy())

snapshots_v = np.array(snapshots_v)
snapshots_w = np.array(snapshots_w)
x = np.linspace(0, L, N + 1)

In [ ]:
def plot_fhn(frame=0):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(x, snapshots_v[frame], color="C0", linewidth=2, label="Voltage (V)")
    ax.plot(
        x,
        snapshots_w[frame],
        color="C1",
        linewidth=2,
        linestyle="--",
        label="Recovery (w)",
    )
    ax.set_xlim(0, L)
    ax.set_ylim(-0.2, 1.2)
    ax.set_xlabel("Position (x)")
    ax.set_ylabel("V / w")
    t_ms = frame * save_every * dt
    ax.set_title(f"FHN Model — t = {t_ms:.1f} ms")
    ax.legend(loc="upper right")
    plt.show()


interact(
    plot_fhn,
    frame=IntSlider(
        min=0, max=len(snapshots_v) - 1, step=1, value=0, description="Time frame"
    ),
)

**Reflection question:** Compare the shape of the FHN wave to the bistable wave from Exercise 16.1. What happens _behind_ the wavefront that did not happen before? What role does the recovery variable $w$ play in shaping the pulse?


## Exercise 17.1b: Periodic boundary conditions

Instead of a straight wire with sealed ends, we want to simulate a ring of tissue (like the circumference of a heart chamber). We can do this mathematically by implementing **periodic boundary conditions**.

This simply means that the "left neighbor" of node $0$ is node $N$, and the "right neighbor" of node $N$ is node $0$!

In NumPy, we can achieve this beautifully without any `if` statements by making `I` cover the _entire_ array, and manually wrapping the ends of our neighbor slice arrays (`Ip` and `Im`). Modify the code below to see the wave travel off the right edge of the screen and instantly reappear on the left!


In [ ]:
# Reset arrays
v = np.zeros(N + 1)
w = np.zeros(N + 1)

# Stimulus in the middle of the cable
mid = int(N / 2)
v[mid - 10 : mid + 10] = 0.3

# Wrap-around slicing arrays!
I = np.arange(N + 1)
Ip = I + 1
Ip[N] = 0  # The right neighbor of the last node is the first node!
Im = I - 1
Im[0] = N  # The left neighbor of the first node is the last node!

n_steps_periodic = 1400
snapshots_v_p = [v.copy()]
snapshots_w_p = [w.copy()]

for i in range(n_steps_periodic):
    v_prev = np.copy(v)
    w_prev = np.copy(w)

    I_ion_v = A * v_prev * (1 - v_prev) * (v_prev - alpha) - w_prev
    I_ion_w = eps * (v_prev - gamma * w_prev)

    # Apply diffusion to the ENTIRE array at once using our wrapped slices!
    v[I] = v_prev[I] + dt * (k / dx**2) * (v_prev[Ip] - 2 * v_prev[I] + v_prev[Im])

    # Add reaction terms
    v = v + dt * I_ion_v
    w = w + dt * I_ion_w

    if (i + 1) % save_every == 0:
        snapshots_v_p.append(v.copy())
        snapshots_w_p.append(w.copy())

snapshots_v_p = np.array(snapshots_v_p)
snapshots_w_p = np.array(snapshots_w_p)

In [ ]:
def plot_periodic(frame=0):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(x, snapshots_v_p[frame], color="C3", linewidth=2, label="Voltage")
    ax.plot(
        x,
        snapshots_w_p[frame],
        color="C4",
        linewidth=2,
        linestyle="--",
        label="Recovery",
    )
    ax.set_xlim(0, L)
    ax.set_ylim(-0.2, 1.2)
    ax.set_xlabel("Position (x)")
    ax.set_ylabel("V / w")
    t_ms = frame * save_every * dt
    ax.set_title(f"Periodic boundaries — t = {t_ms:.1f} ms")
    ax.legend(loc="upper right")
    plt.show()


interact(
    plot_periodic,
    frame=IntSlider(
        min=0, max=len(snapshots_v_p) - 1, step=1, value=0, description="Time frame"
    ),
)

**Question:** When you stimulate the middle of the ring, what happens when the two wavefronts eventually collide on the opposite side? Why does this happen? _(Hint: think about the refractory period.)_


## Exercise 17.1c: Simulating cardiac reentry

A **reentry circuit** occurs when an electrical wave gets trapped traveling in a continuous loop, never stopping. This is the mechanism behind ventricular tachycardia, a dangerous cardiac arrhythmia.

Normally, waves cannot travel backward because the tissue they just passed through is _refractory_ (the $w$ variable is still high, preventing immediate reactivation). Therefore, when two waves collide, they annihilate each other.

To create a reentry circuit, the wave must only travel in **one direction**.

We can force this by creating an artificial "block" on one side of our initial stimulus. We will artificially inject a high $w$ value (refractory tissue) immediately to the left of our stimulus. Copy your periodic boundary code from above, but change the initial conditions to the following:

```python
mid = int(N / 2)
v[mid-10:mid+10] = 0.3      # Stimulate the center
w[:mid-5] = 0.2             # Make the left side completely refractory!
```

Increase the loop range to `14000` steps and run the simulation!


In [ ]:
# Your reentry simulation code here

# When done, use a slider to explore the time evolution
# Example template:
#
# v = np.zeros(N + 1)
# w = np.zeros(N + 1)
#
# mid = int(N / 2)
# v[mid-10:mid+10] = 0.3
# w[:mid-5] = 0.2
#
# ... (run 14000 steps with periodic BCs, saving every 50 steps) ...
#
# Use interact() with IntSlider to scrub through the snapshots.

## Exercise 17.1d: Phase plane exploration (Widget)

To build deeper intuition for the FHN dynamics, we can examine the **phase plane** — a plot of $V$ vs $w$ showing the system's nullclines and trajectory.

The widget below simulates a **single cell** (no spatial diffusion) and lets you adjust the key parameters:

- **$\alpha$**: The excitation threshold
- **$\epsilon$**: Time scale separation (how slowly $w$ responds)
- **$\gamma$**: Recovery decay rate
- **$V_0$**: Initial stimulus strength

**Questions to investigate:**

1. With the default parameters ($\alpha = 0.1$, $\epsilon = 0.005$), set $V_0 = 0.05$ (below threshold). What trajectory does the system take in the phase plane?
2. Now set $V_0 = 0.15$ (above threshold). How does the trajectory differ?
3. Increase $\epsilon$ from 0.005 to 0.05. How does the action potential shape change? Why?
4. What happens if you set $\gamma$ very large (e.g. 10)? Very small (e.g. 0.5)?


In [ ]:
from ipywidgets import interact, FloatSlider


def fhn_phase_plane(alpha_w=0.1, eps_w=0.005, gamma_w=2.0, V0=0.15):
    """Simulate a single FHN cell and show phase plane + time traces."""
    A_w = 1.0
    dt_w = 0.05
    T_total = 600.0  # ms
    n = int(T_total / dt_w)

    # Simulate the ODE
    V_trace = np.zeros(n)
    w_trace = np.zeros(n)
    V_trace[0] = V0
    w_trace[0] = 0.0

    for i in range(1, n):
        V_old = V_trace[i - 1]
        w_old = w_trace[i - 1]
        dV = A_w * V_old * (1 - V_old) * (V_old - alpha_w) - w_old
        dw = eps_w * (V_old - gamma_w * w_old)
        V_trace[i] = V_old + dt_w * dV
        w_trace[i] = w_old + dt_w * dw

    # Nullclines
    V_nc = np.linspace(-0.2, 1.2, 300)
    w_V_nullcline = A_w * V_nc * (1 - V_nc) * (V_nc - alpha_w)  # dV/dt = 0
    w_w_nullcline = V_nc / gamma_w  # dw/dt = 0

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Phase plane
    ax = axes[0]
    ax.plot(V_nc, w_V_nullcline, "b-", linewidth=2, label="V-nullcline (dV/dt=0)")
    ax.plot(V_nc, w_w_nullcline, "r-", linewidth=2, label="w-nullcline (dw/dt=0)")
    ax.plot(V_trace, w_trace, "k-", linewidth=1, alpha=0.7, label="Trajectory")
    ax.plot(V_trace[0], w_trace[0], "go", markersize=10, label="Start")
    ax.plot(V_trace[-1], w_trace[-1], "rs", markersize=8, label="End")
    ax.set_xlim(-0.3, 1.3)
    ax.set_ylim(-0.05, 0.3)
    ax.set_xlabel("V (voltage)")
    ax.set_ylabel("w (recovery)")
    ax.set_title("Phase plane")
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, alpha=0.3)

    # Time traces
    ax2 = axes[1]
    t_arr = np.linspace(0, T_total, n)
    ax2.plot(t_arr, V_trace, "C0", linewidth=2, label="V(t)")
    ax2.plot(t_arr, w_trace, "C1", linewidth=2, linestyle="--", label="w(t)")
    ax2.set_xlabel("Time (ms)")
    ax2.set_ylabel("V / w")
    ax2.set_title("Single-cell time traces")
    ax2.legend(loc="upper right")
    ax2.set_ylim(-0.3, 1.3)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


interact(
    fhn_phase_plane,
    alpha_w=FloatSlider(min=0.05, max=0.45, step=0.01, value=0.10, description="α"),
    eps_w=FloatSlider(
        min=0.001,
        max=0.10,
        step=0.001,
        value=0.005,
        description="ε",
        readout_format=".3f",
    ),
    gamma_w=FloatSlider(min=0.5, max=10.0, step=0.1, value=2.0, description="γ"),
    V0=FloatSlider(min=0.0, max=0.5, step=0.01, value=0.15, description="V₀"),
);